In [11]:
# ============================================================
# Boosting Model Comparison with Tuning, Pipelines & ColumnTransformer
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

# ── 1. Load & Prepare Data ──────────────────────────────────
df = pd.read_csv('boxoffice.csv')
df = df.drop(columns=['title'])

X = df.drop(columns=['world_revenue'])
y = df['world_revenue']

# ── 2. Define Column Groups ─────────────────────────────────
numeric_features = ['domestic_revenue', 'opening_revenue', 'opening_theaters', 'budget', 'release_days']
categorical_features = ['distributor', 'MPAA', 'genres']

# ── 3. Column Transformer ───────────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

# ── 4. Models & Hyperparameter Grids ────────────────────────
# Keys must be prefixed with 'regressor__' because they're inside a Pipeline

models_and_params = {
    'GradientBoosting': {
        'model': GradientBoostingRegressor(random_state=42),
        'params': {
            'regressor__n_estimators': [100, 200, 300, 500],
            'regressor__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'regressor__max_depth': [3, 5, 7, 9],
            'regressor__min_samples_split': [2, 5, 10],
            'regressor__min_samples_leaf': [1, 2, 4],
            'regressor__subsample': [0.7, 0.8, 0.9, 1.0],
        }
    },
    'AdaBoost': {
        'model': AdaBoostRegressor(random_state=42),
        'params': {
            'regressor__n_estimators': [50, 100, 200, 300, 500],
            'regressor__learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0],
            'regressor__loss': ['linear', 'square', 'exponential'],
        }
    },
    'XGBoost': {
        'model': XGBRegressor(random_state=42, verbosity=0),
        'params': {
            'regressor__n_estimators': [100, 200, 300, 500],
            'regressor__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'regressor__max_depth': [3, 5, 7, 9],
            'regressor__min_child_weight': [1, 3, 5],
            'regressor__subsample': [0.7, 0.8, 0.9, 1.0],
            'regressor__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
            'regressor__reg_alpha': [0, 0.01, 0.1, 1],
            'regressor__reg_lambda': [0.5, 1, 1.5, 2],
        }
    },
    'LightGBM': {
        'model': LGBMRegressor(random_state=42, verbose=-1),
        'params': {
            'regressor__n_estimators': [100, 200, 300, 500],
            'regressor__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'regressor__max_depth': [3, 5, 7, 10, -1],
            'regressor__num_leaves': [15, 31, 63, 127],
            'regressor__min_child_samples': [5, 10, 20, 30],
            'regressor__subsample': [0.7, 0.8, 0.9, 1.0],
            'regressor__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
            'regressor__reg_alpha': [0, 0.01, 0.1, 1],
            'regressor__reg_lambda': [0, 0.01, 0.1, 1],
        }
    },
}

# ── 5. Train-Test Split ─────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 6. Tune & Evaluate Each Model ───────────────────────────
results = []
best_models = {}

for name, config in models_and_params.items():
    print(f"⏳ Tuning {name}...")
    
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', config['model'])
    ])
    
    search = RandomizedSearchCV(
        pipe,
        param_distributions=config['params'],
        n_iter=50,           # 50 random combinations per model
        cv=5,
        scoring='r2',
        random_state=42,
        n_jobs=-1,
        return_train_score=True
    )
    
    search.fit(X_train, y_train)
    
    # Best estimator predictions
    y_pred = search.best_estimator_.predict(X_val)
    
    results.append({
        'Model': name,
        'Best CV R²': round(search.best_score_, 4),
        'Val R²': round(r2_score(y_val, y_pred), 4),
        'MAE': round(mean_absolute_error(y_val, y_pred), 2),
        'RMSE': round(np.sqrt(mean_squared_error(y_val, y_pred)), 2),
    })
    
    best_models[name] = search
    print(f"   ✅ {name} done — Best CV R²: {search.best_score_:.4f}")

# ── 7. Results Comparison ────────────────────────────────────
results_df = pd.DataFrame(results).sort_values('Val R²', ascending=False)
print("\n" + "=" * 75)
print("📊  TUNED BOOSTING MODEL COMPARISON")
print("=" * 75)
display(results_df.reset_index(drop=True))

# ── 8. Best Overall Model ───────────────────────────────────
best_name = results_df.iloc[0]['Model']
print(f"\n🏆 Best Model: {best_name}\n")

# ── 9. Best Hyperparameters for Each Model ───────────────────
print("=" * 75)
print("⚙️  BEST HYPERPARAMETERS (after tuning)")
print("=" * 75)
for name, search in best_models.items():
    print(f"\n▸ {name}:")
    best_params = search.best_params_
    for k, v in sorted(best_params.items()):
        # Remove 'regressor__' prefix for cleaner display
        clean_key = k.replace('regressor__', '')
        print(f"    {clean_key}: {v}")


⏳ Tuning GradientBoosting...
   ✅ GradientBoosting done — Best CV R²: -0.0096
⏳ Tuning AdaBoost...
   ✅ AdaBoost done — Best CV R²: -0.0007
⏳ Tuning XGBoost...
   ✅ XGBoost done — Best CV R²: -0.0086
⏳ Tuning LightGBM...
   ✅ LightGBM done — Best CV R²: -0.0105

📊  TUNED BOOSTING MODEL COMPARISON


,Model,Best CV R²,Val R²,MAE,RMSE
0,XGBoost,-0.0086,0.0031,3.708084e+08,4.299845e+08
1,GradientBoosting,-0.0096,-0.0002,3.722620e+08,4.306935e+08
2,AdaBoost,-0.0007,-0.0045,3.726585e+08,4.316254e+08
3,LightGBM,-0.0105,-0.0078,3.730278e+08,4.323207e+08



🏆 Best Model: XGBoost

⚙️  BEST HYPERPARAMETERS (after tuning)

▸ GradientBoosting:
    learning_rate: 0.01
    max_depth: 5
    min_samples_leaf: 2
    min_samples_split: 5
    n_estimators: 100
    subsample: 0.7

▸ AdaBoost:
    learning_rate: 0.5
    loss: exponential
    n_estimators: 100

▸ XGBoost:
    colsample_bytree: 0.9
    learning_rate: 0.01
    max_depth: 5
    min_child_weight: 3
    n_estimators: 100
    reg_alpha: 1
    reg_lambda: 1.5
    subsample: 0.9

▸ LightGBM:
    colsample_bytree: 0.9
    learning_rate: 0.01
    max_depth: 7
    min_child_samples: 30
    n_estimators: 100
    num_leaves: 31
    reg_alpha: 0.1
    reg_lambda: 0
    subsample: 1.0
